In [1]:
import pandas as pd
import sqlite3

# Connect to the database
conn = sqlite3.connect('../data/flights.db')

print("Connected to flights database")

Connected to flights database


In [4]:
# Worst Airline for departure delays
query = """
SELECT 
    AIRLINE_CODE,
    AIRLINE,
    COUNT(*) AS total_flights,
    ROUND(AVG(DEP_DELAY), 2) AS avg_dep_delay
FROM flights
WHERE DEP_DELAY IS NOT NULL
GROUP BY AIRLINE_CODE, AIRLINE
ORDER BY avg_dep_delay DESC
LIMIT 10;
"""

# Run the query and get results back as a pandas DataFrame
result = pd.read_sql_query(query, conn)
result

,AIRLINE_CODE,AIRLINE,total_flights,avg_dep_delay
0,B6,JetBlue Airways,109882,18.32
1,F9,Frontier Airlines Inc.,62846,16.03
2,G4,Allegiant Air,50367,13.91
3,NK,Spirit Air Lines,93433,12.98
4,EV,ExpressJet Airlines LLC d/b/a aha!,18028,12.77
5,AA,American Airlines Inc.,372441,12.61
6,YV,Mesa Airlines Inc.,62686,12.28
7,UA,United Air Lines Inc.,249083,11.22
8,WN,Southwest Airlines Co.,557105,10.82
9,OO,SkyWest Airlines Inc.,336153,9.46


In [5]:
# Days of the week in relation to average departure and arrival delays

query = """
SELECT 
    CASE strftime('%w', FL_DATE)
        WHEN '0' THEN 'Sunday'
        WHEN '1' THEN 'Monday'
        WHEN '2' THEN 'Tuesday'
        WHEN '3' THEN 'Wednesday'
        WHEN '4' THEN 'Thursday'
        WHEN '5' THEN 'Friday'
        WHEN '6' THEN 'Saturday'
    END AS day_of_week,
    COUNT(*) AS total_flights,
    ROUND(AVG(DEP_DELAY), 2) AS avg_dep_delay,
    ROUND(AVG(ARR_DELAY), 2) AS avg_arr_delay
FROM flights
WHERE DEP_DELAY IS NOT NULL
GROUP BY strftime('%w', FL_DATE)
ORDER BY strftime('%w', FL_DATE);
"""

result = pd.read_sql_query(query, conn)
result

,day_of_week,total_flights,avg_dep_delay,avg_arr_delay
0,Sunday,425149,11.26,5.29
1,Monday,434863,10.59,4.72
2,Tuesday,406458,7.99,1.69
3,Wednesday,411714,8.44,2.63
4,Thursday,434529,10.80,5.56
5,Friday,434892,11.39,5.95
6,Saturday,374751,10.19,3.67


In [6]:
# Worst Airports for departure delays

query = """
SELECT 
    ORIGIN, 
    ORIGIN_CITY, 
    COUNT(*) AS total_flights, 
    ROUND(AVG(DEP_DELAY), 2) AS avg_dep_delay  
FROM flights 
WHERE DEP_DELAY IS NOT NULL 
GROUP BY ORIGIN, ORIGIN_CITY 
ORDER BY avg_dep_delay DESC 
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)
result

,ORIGIN,ORIGIN_CITY,total_flights,avg_dep_delay
0,PPG,"Pago Pago, TT",31,58.97
1,SMX,"Santa Maria, CA",86,36.27
2,PSM,"Portsmouth, NH",165,31.02
3,PGV,"Greenville, NC",28,28.89
4,HGR,"Hagerstown, MD",89,27.57
5,PIB,"Hattiesburg/Laurel, MS",304,26.05
6,ASE,"Aspen, CO",2589,25.89
7,ART,"Watertown, NY",18,25.44
8,OGS,"Ogdensburg, NY",216,24.56
9,HYA,"Hyannis, MA",47,24.43


In [7]:
# Worst Airports for departure delays revised (due to sample size problem)
#Filtering to only meaningful airports, and setting flight count threshold to 10,000+ before ranking 

query = """
SELECT 
    ORIGIN, 
    ORIGIN_CITY, 
    COUNT(*) AS total_flights, 
    ROUND(AVG(DEP_DELAY), 2) AS avg_dep_delay  
FROM flights 
WHERE DEP_DELAY IS NOT NULL 
GROUP BY ORIGIN, ORIGIN_CITY 
HAVING COUNT(*) >= 10000
ORDER BY avg_dep_delay DESC 
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)
result



,ORIGIN,ORIGIN_CITY,total_flights,avg_dep_delay
0,EWR,"Newark, NJ",50991,16.06
1,FLL,"Fort Lauderdale, FL",39210,14.67
2,SJU,"San Juan, PR",12773,14.64
3,MCO,"Orlando, FL",62168,14.57
4,MDW,"Chicago, IL",33666,13.79
5,MIA,"Miami, FL",41062,13.53
6,DEN,"Denver, CO",116726,13.42
7,JFK,"New York, NY",49107,13.32
8,PBI,"West Palm Beach/Palm Beach, FL",10639,13.24
9,BWI,"Baltimore, MD",39676,12.93


In [8]:
# Worst Airports for departure delays revised for larger threshold (test)
#Filtering to only meaningful airports, But setting flight count threshold to 50,000+ before ranking 

query = """
SELECT 
    ORIGIN, 
    ORIGIN_CITY, 
    COUNT(*) AS total_flights, 
    ROUND(AVG(DEP_DELAY), 2) AS avg_dep_delay  
FROM flights 
WHERE DEP_DELAY IS NOT NULL 
GROUP BY ORIGIN, ORIGIN_CITY 
HAVING COUNT(*) >= 50000
ORDER BY avg_dep_delay DESC 
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)
result

,ORIGIN,ORIGIN_CITY,total_flights,avg_dep_delay
0,EWR,"Newark, NJ",50991,16.06
1,MCO,"Orlando, FL",62168,14.57
2,DEN,"Denver, CO",116726,13.42
3,DFW,"Dallas/Fort Worth, TX",126254,12.68
4,LAS,"Las Vegas, NV",71599,12.16
5,LGA,"New York, NY",60149,12.04
6,ORD,"Chicago, IL",118601,11.67
7,IAH,"Houston, TX",61081,11.24
8,BOS,"Boston, MA",53907,11.17
9,DCA,"Washington, DC",51581,9.73


In [9]:
# Delay Cause 

query = """
SELECT 
    COUNT(*) AS total_delayed_flights,
    ROUND(AVG(DELAY_DUE_CARRIER), 2) AS avg_carrier_delay,
    ROUND(AVG(DELAY_DUE_WEATHER), 2) AS avg_weather_delay,
    ROUND(AVG(DELAY_DUE_NAS), 2) AS avg_nas_delay,
    ROUND(AVG(DELAY_DUE_SECURITY), 2) AS avg_security_delay,
    ROUND(AVG(DELAY_DUE_LATE_AIRCRAFT), 2) AS avg_late_aircraft_delay
FROM flights
WHERE ARR_DELAY >= 15;
"""

result = pd.read_sql_query(query, conn)
result

,total_delayed_flights,avg_carrier_delay,avg_weather_delay,avg_nas_delay,avg_security_delay,avg_late_aircraft_delay
0,533863,24.76,3.99,13.16,0.15,25.47


In [10]:
# sanity check queury to verify 15 min thershold

verify_query = """
SELECT 
    COUNT(*) AS total_flights,
    SUM(CASE WHEN ARR_DELAY >= 15 THEN 1 ELSE 0 END) AS delayed_15plus,
    SUM(CASE WHEN DELAY_DUE_CARRIER IS NOT NULL THEN 1 ELSE 0 END) AS has_cause_data
FROM flights;
"""
pd.read_sql_query(verify_query, conn)

,total_flights,delayed_15plus,has_cause_data
0,3000000,533863,533863


In [11]:
# Cancellation Rates by airline 

query = """
SELECT 
    AIRLINE_CODE,
    AIRLINE,
    COUNT(*) AS total_flights,
    SUM(CANCELLED) AS cancelled_flights,
    ROUND(AVG(CANCELLED) * 100, 2) AS cancellation_rate_pct
FROM flights
GROUP BY AIRLINE_CODE, AIRLINE
HAVING COUNT(*) >= 10000
ORDER BY cancellation_rate_pct DESC;
"""

result = pd.read_sql_query(query, conn)
result

,AIRLINE_CODE,AIRLINE,total_flights,cancelled_flights,cancellation_rate_pct
0,EV,ExpressJet Airlines LLC d/b/a aha!,19082,1062.0,5.57
1,G4,Allegiant Air,52738,2383.0,4.52
2,YV,Mesa Airlines Inc.,65012,2373.0,3.65
3,WN,Southwest Airlines Co.,576470,19465.0,3.38
4,YX,Republic Airline,143107,4646.0,3.25
5,OH,PSA Airlines Inc.,107050,3301.0,3.08
6,MQ,Envoy Air,121256,3633.0,3.00
7,AA,American Airlines Inc.,383106,10907.0,2.85
8,B6,JetBlue Airways,112844,3039.0,2.69
9,F9,Frontier Airlines Inc.,64466,1666.0,2.58


In [12]:
# Seasonal Patterns by month 

query = """
SELECT 
    CAST(strftime('%m', FL_DATE) AS INTEGER) AS month_num,
    CASE strftime('%m', FL_DATE)
        WHEN '01' THEN 'January'
        WHEN '02' THEN 'February'
        WHEN '03' THEN 'March'
        WHEN '04' THEN 'April'
        WHEN '05' THEN 'May'
        WHEN '06' THEN 'June'
        WHEN '07' THEN 'July'
        WHEN '08' THEN 'August'
        WHEN '09' THEN 'September'
        WHEN '10' THEN 'October'
        WHEN '11' THEN 'November'
        WHEN '12' THEN 'December'
    END AS month_name,
    COUNT(*) AS total_flights,
    ROUND(AVG(DEP_DELAY), 2) AS avg_dep_delay,
    ROUND(AVG(CANCELLED) * 100, 2) AS cancellation_rate_pct
FROM flights
WHERE DEP_DELAY IS NOT NULL OR CANCELLED = 1
GROUP BY month_num, month_name
ORDER BY month_num;
"""

result = pd.read_sql_query(query, conn)
result

,month_num,month_name,total_flights,avg_dep_delay,cancellation_rate_pct
0,1,January,268702,8.72,2.73
1,2,February,248497,9.82,2.89
2,3,March,293099,8.32,5.03
3,4,April,253917,9.60,6.85
4,5,May,252530,9.43,1.72
5,6,June,261136,15.05,2.01
6,7,July,284968,14.67,1.79
7,8,August,287104,11.85,1.99
8,9,September,206928,5.78,1.28
9,10,October,216790,7.33,1.08


In [13]:
# since high cancellation in April an march, possible due to COVID in 2020 and high flight cancellations
# Verify COVID hypothesis for march and april spike

verify_covid = """
SELECT 
    strftime('%Y', FL_DATE) AS year,
    ROUND(AVG(CANCELLED) * 100, 2) AS cancellation_rate_pct
FROM flights
WHERE strftime('%m', FL_DATE) IN ('03', '04')
GROUP BY year
ORDER BY year;
"""
pd.read_sql_query(verify_covid, conn)

,year,cancellation_rate_pct
0,2019,2.21
1,2020,24.94
2,2021,0.87
3,2022,1.94
4,2023,1.54


In [3]:
import scipy.stats as stats

# Get weekend delays (Saturday=6, Sunday=0)
weekend_query = """
SELECT DEP_DELAY
FROM flights
WHERE DEP_DELAY IS NOT NULL
  AND strftime('%w', FL_DATE) IN ('0', '6');
"""

# Get weekday delays (Monday=1 through Friday=5)
weekday_query = """
SELECT DEP_DELAY
FROM flights
WHERE DEP_DELAY IS NOT NULL
  AND strftime('%w', FL_DATE) IN ('1', '2', '3', '4', '5');
"""

print("Loading weekend delays...")
weekend_delays = pd.read_sql_query(weekend_query, conn)['DEP_DELAY'].values

print("Loading weekday delays...")
weekday_delays = pd.read_sql_query(weekday_query, conn)['DEP_DELAY'].values

print(f"\nWeekend flights: {len(weekend_delays):,}")
print(f"Weekday flights: {len(weekday_delays):,}")
print(f"\nWeekend mean delay: {weekend_delays.mean():.2f} min")
print(f"Weekday mean delay: {weekday_delays.mean():.2f} min")

Loading weekend delays...
Loading weekday delays...

Weekend flights: 799,900
Weekday flights: 2,122,456

Weekend mean delay: 10.76 min
Weekday mean delay: 9.88 min


In [4]:
# Run independent two-sample t-test  (not assuming equal variances)
t_statistic, p_value = stats.ttest_ind(
    weekend_delays, 
    weekday_delays, 
    equal_var=False  # Welch's t-test because safer when group sizes/variances differ
)

print(f"T-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.4e}")  # scientific notation since p will likely be very small
print(f"\nWeekend mean: {weekend_delays.mean():.4f} min")
print(f"Weekday mean: {weekday_delays.mean():.4f} min")
print(f"Difference: {weekend_delays.mean() - weekday_delays.mean():.4f} min")

# Standard deviations
print(f"\nWeekend std dev: {weekend_delays.std():.2f} min")
print(f"Weekday std dev: {weekday_delays.std():.2f} min")

# Effect size  (Cohen's d)
import numpy as np
pooled_std = np.sqrt(((len(weekend_delays) - 1) * weekend_delays.var() + 
                     (len(weekday_delays) - 1) * weekday_delays.var()) / 
                    (len(weekend_delays) + len(weekday_delays) - 2))
cohens_d = (weekend_delays.mean() - weekday_delays.mean()) / pooled_std
print(f"\nCohen's d (effect size): {cohens_d:.4f}")

T-statistic: 13.3098
P-value: 2.0422e-40

Weekend mean: 10.7631 min
Weekday mean: 9.8822 min
Difference: 0.8809 min

Weekend std dev: 51.16 min
Weekday std dev: 48.51 min

Cohen's d (effect size): 0.0179


## Statistical Test: Weekend vs Weekday Delays

**Hypothesis test:** Welch's two-sample t-test comparing departure delays.

**Results:**
- t-statistic: 13.31
- p-value: 2.04 × 10⁻⁴⁰ (statistically significant)
- Cohen's d: 0.018 (practically negligible)

**Interpretation:** Although the difference between weekend and weekday delays is statistically significant due to the very large sample size, the effect is extremely small (Cohen's d = 0.018, far below the 0.2 threshold for a "small" effect). The 0.88-minute mean difference is dwarfed by within-group variation (std dev ≈ 50 minutes). Day-of-week is not a useful standalone predictor of delay for operational decisions.

In [3]:
# Extract data for ML model
# Features: airline, origin, destination, day of week, hour, month
# Label: 1 if ARR_DELAY >= 15, else 0
# Filter: exclude cancelled flights (no delay data)

ml_query = """
SELECT 
    AIRLINE_CODE,
    ORIGIN,
    DEST,
    CAST(strftime('%w', FL_DATE) AS INTEGER) AS day_of_week,
    CAST(strftime('%m', FL_DATE) AS INTEGER) AS month,
    CAST(CRS_DEP_TIME / 100 AS INTEGER) AS scheduled_hour,
    CASE WHEN ARR_DELAY >= 15 THEN 1 ELSE 0 END AS is_delayed
FROM flights
WHERE ARR_DELAY IS NOT NULL
  AND CRS_DEP_TIME IS NOT NULL;
"""

print("Loading ML dataset from SQLite...")
ml_df = pd.read_sql_query(ml_query, conn)
print(f"Loaded {len(ml_df):,} rows")
print(f"\nFirst 5 rows:")
ml_df.head()

Loading ML dataset from SQLite...
Loaded 2,913,802 rows

First 5 rows:


,AIRLINE_CODE,ORIGIN,DEST,day_of_week,month,scheduled_hour,is_delayed
0,UA,FLL,EWR,3,1,11,0
1,DL,MSP,SEA,6,11,21,0
2,UA,DEN,MSP,5,7,9,0
3,DL,MSP,SFO,1,3,16,1
4,NK,MCO,DFW,0,2,18,0


In [4]:
# Verify class balance
class_balance = ml_df['is_delayed'].value_counts(normalize=True)
print("Class distribution:")
print(class_balance)
print(f"\nDelayed (1): {ml_df['is_delayed'].sum():,} flights")
print(f"On-time (0): {(ml_df['is_delayed'] == 0).sum():,} flights")

Class distribution:
is_delayed
0    0.816781
1    0.183219
Name: proportion, dtype: float64

Delayed (1): 533,863 flights
On-time (0): 2,379,939 flights


# Flight Delay Analysis — SQL Queries + ML Setup

This notebook contains the SQL analysis and ML data preparation for the flight delay project, using the US BTS 3M-row sample (2019-2023).

## Contents
1. **Setup** — pandas, sqlite3 imports, database connection
2. **Query 1** — Top 10 worst airlines by avg departure delay
3. **Query 2** — Day-of-week delay patterns
4. **Query 3** — Worst airports (with sample-size sensitivity analysis at 10k and 50k thresholds)
5. **Query 4** — Breakdown of delay causes
6. **Query 5A** — Cancellation rates by airline
7. **Query 5B** — Seasonal patterns by month
8. **Diagnostic** — 2020 COVID effect on cancellation rates
9. **Charts** — 5 matplotlib visualizations saved to `../charts/`
10. **Phase 6: Statistical test** — Welch's t-test on weekend vs weekday delays
11. **Phase 7 Setup** — ML data extraction and class balance verification (next: train/test split and model fitting)

In [7]:
from sklearn.model_selection import train_test_split

# Separate features (X) from label (y)
feature_columns = ['AIRLINE_CODE', 'ORIGIN', 'DEST', 'day_of_week', 'month', 'scheduled_hour']
X = ml_df[feature_columns]
y = ml_df['is_delayed']

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Label distribution: {y.value_counts(normalize=True).round(4).to_dict()}")

Features shape: (2913802, 6)
Labels shape: (2913802,)
Label distribution: {0: 0.8168, 1: 0.1832}


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix
import numpy as np

# Step A: Train/test split before encoding
# Before in order to avoid "data leakage" — fitting the encoder on test data would
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,        # 20% held out for testing
    random_state=42,       # reproducibility (same split every run)
    stratify=y             # preserve class balance in both splits
)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set:     {X_test.shape[0]:,} rows")
print(f"Train class balance: {y_train.value_counts(normalize=True).round(4).to_dict()}")
print(f"Test class balance:  {y_test.value_counts(normalize=True).round(4).to_dict()}")

Training set: 2,331,041 rows
Test set:     582,761 rows
Train class balance: {0: 0.8168, 1: 0.1832}
Test class balance:  {0: 0.8168, 1: 0.1832}


In [9]:
# Verify cardinality before encoding
print(f"Unique airlines:     {ml_df['AIRLINE_CODE'].nunique()}")
print(f"Unique origins:      {ml_df['ORIGIN'].nunique()}")
print(f"Unique destinations: {ml_df['DEST'].nunique()}")
print(f"Unique day_of_week:  {ml_df['day_of_week'].nunique()}")
print(f"Unique month:        {ml_df['month'].nunique()}")
print(f"Unique scheduled_hour: {ml_df['scheduled_hour'].nunique()}")

Unique airlines:     18
Unique origins:      380
Unique destinations: 380
Unique day_of_week:  7
Unique month:        12
Unique scheduled_hour: 24


In [10]:
from sklearn.preprocessing import OneHotEncoder

# Identify categorical columns to encode
categorical_cols = ['AIRLINE_CODE', 'ORIGIN', 'DEST', 'day_of_week', 'month', 'scheduled_hour']

# Create the encoder
# handle_unknown='ignore' — if test set has a category not seen in train, encode as all zeros 
# sparse_output=True — return sparse matrix in order to save memory
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)

# Fit on training data ONLY
print("Fitting encoder on training data...")
encoder.fit(X_train[categorical_cols])

# Transform both train and test
print("Encoding training set...")
X_train_encoded = encoder.transform(X_train[categorical_cols])

print("Encoding test set...")
X_test_encoded = encoder.transform(X_test[categorical_cols])

print(f"\nEncoded training shape: {X_train_encoded.shape}")
print(f"Encoded test shape:     {X_test_encoded.shape}")
print(f"Sparsity: {1 - X_train_encoded.nnz / (X_train_encoded.shape[0] * X_train_encoded.shape[1]):.4%}")

Fitting encoder on training data...
Encoding training set...
Encoding test set...

Encoded training shape: (2331041, 821)
Encoded test shape:     (582761, 821)
Sparsity: 99.2692%


In [11]:
from sklearn.linear_model import LogisticRegression
import time

# Initialize the model
model = LogisticRegression(
    class_weight='balanced',  # handle class imbalance
    max_iter=1000,            # allow enough iterations to converge
    solver='lbfgs',           # default, good for this problem size
    random_state=42,          # reproducibility
    n_jobs=-1                 # use all CPU cores
)

# Fit the model 
print("Training logistic regression model on 2.3M rows × 821 features...")
print("This will take a few minutes. Don't interrupt.\n")

start_time = time.time()
model.fit(X_train_encoded, y_train)
elapsed = time.time() - start_time

print(f"Training complete in {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")
print(f"Number of iterations: {model.n_iter_[0]}")

# Verification — what does the model output for the first few test examples?
sample_predictions = model.predict_proba(X_test_encoded[:5])
print(f"\nSample predictions (first 5 test rows):")
print(f"  Probability of being on-time:  {sample_predictions[:, 0]}")
print(f"  Probability of being delayed:  {sample_predictions[:, 1]}")

Training logistic regression model on 2.3M rows × 821 features...
This will take a few minutes. Don't interrupt.



C:\Users\elias\Desktop\Flight Analysis Project\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Training complete in 6.2 seconds (0.1 minutes)
Number of iterations: 34

Sample predictions (first 5 test rows):
  Probability of being on-time:  [0.5340355  0.61624953 0.29120806 0.41515128 0.5853935 ]
  Probability of being delayed:  [0.4659645  0.38375047 0.70879194 0.58484872 0.4146065 ]


In [12]:
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Predict on test set
print("Generating predictions on test set...")
y_pred = model.predict(X_test_encoded)
y_pred_proba = model.predict_proba(X_test_encoded)[:, 1]  # probability of delay class

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n--- Model Performance on Test Set ---")
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}")

print(f"\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_pred)
print(f"                    Predicted On-Time  Predicted Delayed")
print(f"Actual On-Time      {cm[0,0]:>15,}  {cm[0,1]:>17,}")
print(f"Actual Delayed      {cm[1,0]:>15,}  {cm[1,1]:>17,}")

print(f"\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['On-Time', 'Delayed']))

Generating predictions on test set...

--- Model Performance on Test Set ---
Accuracy:  0.5931 (59.31%)
Precision: 0.2526
Recall:    0.6233
F1 Score:  0.3595
ROC AUC:   0.6449

--- Confusion Matrix ---
                    Predicted On-Time  Predicted Delayed
Actual On-Time              279,104            196,884
Actual Delayed               40,219             66,554

--- Classification Report ---
              precision    recall  f1-score   support

     On-Time       0.87      0.59      0.70    475988
     Delayed       0.25      0.62      0.36    106773

    accuracy                           0.59    582761
   macro avg       0.56      0.60      0.53    582761
weighted avg       0.76      0.59      0.64    582761



## Phase 7: Predictive Model — Logistic Regression

**Goal:** Predict whether a flight will be delayed ≥15 minutes (BTS delay threshold) using features available at scheduling time.

**Features (6):** Airline code, origin airport, destination airport, day of week, month, scheduled hour.

**Model:** Logistic regression with class_weight='balanced' to handle the 81.7%/18.3% imbalance.

**Training:** 2,331,041 rows. Test: 582,761 rows. Stratified split.

### Results on Held-Out Test Set

| Metric | Value | Interpretation |
|---|---|---|
| ROC AUC | 0.6449 | Weak but informative — model learns real signal |
| Recall (delayed) | 0.6233 | Catches 62% of actual delays |
| Precision (delayed) | 0.2526 | 75% of delay predictions are false alarms |
| F1 (delayed) | 0.3595 | Mediocre on the minority class |
| Accuracy | 0.5931 | Below naive baseline of 0.8168 |

### Honest Interpretation

The class_weight='balanced' setting deliberately trades overall accuracy for higher recall on the delayed class. The naive "always predict on-time" baseline achieves 81.68% accuracy but catches zero delays; this model catches 62% of delays at the cost of overall accuracy.

The ROC AUC of 0.64 indicates the model has learned real signal but the predictive power is limited. This is a realistic finding given our feature set: scheduling-time features (route, time, carrier) can only capture average delay tendencies. Production delay prediction systems rely on features unavailable to us:

- Real-time weather conditions at origin and destination
- Current airport congestion and ATC delays  
- Inbound aircraft status (late-arriving aircraft cascade)
- Crew availability and equipment status
- NAS (National Airspace System) advisories

### Future Improvements

1. Integrate real-time operational data (weather APIs, FAA NAS status)
2. Engineer route-level features (historical congestion, route distance bins)
3. Try gradient boosting (XGBoost/LightGBM) for non-linear interactions
4. Tune classification threshold for specific business objectives (precision vs recall tradeoff)